# Author GCN — Table 3 & Table 4

Chỉ đánh giá checkpoint trong Kaggle Dataset `author_hc3`; không train. Bỏ GPT-3.5 Mixed 0% vì đã có trong notebook train. Table 3 chạy synonym 5/10/20/30%; Table 4 chạy HC3 Original và Insert/Repeat/Replace/Delete 25%.

Bật Internet + GPU T4/P100. Builder tác giả chạy RoBERTa CPU nên giai đoạn tạo graph chậm; suy luận GCN chạy GPU và batch 1 như `test.py`.

In [ ]:
from pathlib import Path
import re, subprocess
WORK=Path('/kaggle/working/author_gcn_table3_table4'); AUTHOR=WORK/'PRDetect'; ENV=WORK/'python38'; PYTHON=ENV/'bin/python'; GRAPH=WORK/'graph_data'; RUN=WORK/'run'
COMMIT='83ebc6e2cd543477f83e405d702ac837e664fc9e'
CKPT=Path('/kaggle/input/author_hc3')
if not CKPT.is_dir():
    choices=[d for d in Path('/kaggle/input').iterdir() if d.is_dir() and list(d.rglob('*.pth'))]
    assert len(choices)==1, choices
    CKPT=choices[0]
CHECKPOINTS=sorted(CKPT.rglob('*.pth'))
assert CHECKPOINTS, f'No .pth in {CKPT}'
seeds=[]
for p in CHECKPOINTS:
    m=re.search(r'(20\d{2})(?:\.pth)?$',p.name); assert m, p.name; seeds.append(int(m.group(1)))
assert len(seeds)==len(set(seeds)), seeds
WORK.mkdir(parents=True,exist_ok=True); GRAPH.mkdir(exist_ok=True); RUN.mkdir(exist_ok=True)
assert not (RUN/'author_table3_table4_results.zip').exists(), 'Đổi RUN để không ghi đè kết quả cũ.'
%pip install -q uv
!uv python install 3.8.20
if not PYTHON.is_file():
    subprocess.run(["uv", "venv", "--python", "3.8.20", "--seed", str(ENV)], check=True)
!uv pip install --python "{PYTHON}" --no-cache "https://download.pytorch.org/whl/cu113/torch-1.11.0%2Bcu113-cp38-cp38-linux_x86_64.whl"
!uv pip install --python "{PYTHON}" --no-cache "numpy==1.22.4" "scipy==1.10.1" "scikit-learn==1.3.1" "spacy==3.7.4" "transformers==4.40.0" "huggingface-hub==0.20.1" "tqdm==4.66.1" "protobuf==3.19.6" "requests==2.31.0"
!uv pip install --python "{PYTHON}" --no-cache --no-index --only-binary=:all: -f "https://data.pyg.org/whl/torch-1.11.0+cu113.html" "torch-scatter==2.0.9" "torch-sparse==0.6.13"
!uv pip install --python "{PYTHON}" --no-cache "torch-geometric==2.5.2" "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
assert _exit_code==0
if not (AUTHOR / '.git').is_dir():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/thulx18/PRDetect.git", str(AUTHOR)], check=True)
    subprocess.run(["git", "-C", str(AUTHOR), "checkout", "--detach", COMMIT], check=True)
assert subprocess.check_output(['git','-C',str(AUTHOR),'rev-parse','HEAD'],text=True).strip()==COMMIT
print('checkpoints:',[(p.name,s) for p,s in zip(CHECKPOINTS,seeds)])

In [ ]:
%%writefile /kaggle/working/author_gcn_table3_table4/run_eval.py
import argparse,csv,hashlib,json,pickle,re,shutil,subprocess,sys,time
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score
from torch_geometric.data import Data
from huggingface_hub import snapshot_download

p=argparse.ArgumentParser();p.add_argument('--author',type=Path,required=True);p.add_argument('--graphs',type=Path,required=True);p.add_argument('--ckpt',type=Path,required=True);p.add_argument('--run',type=Path,required=True);a=p.parse_args()
CASES=[
('table3','Synonym 5%',5,'gpt3.5_mixed_test_adj_5_synonym_replace','perturbed_text/gpt3.5_mixed_test_adj_5_synonym_replace.json'),
('table3','Synonym 10%',10,'gpt3.5_mixed_test_adj_10_synonym_replace','perturbed_text/gpt3.5_mixed_test_adj_10_synonym_replace.json'),
('table3','Synonym 20%',20,'gpt3.5_mixed_test_adj_20_synonym_replace','perturbed_text/gpt3.5_mixed_test_adj_20_synonym_replace.json'),
('table3','Synonym 30%',30,'gpt3.5_mixed_test_adj_30_synonym_replace','perturbed_text/gpt3.5_mixed_test_adj_30_synonym_replace.json'),
('table4','Original',0,'hc3_test','original_text/hc3_test.json'),
('table4','Insert 25%',25,'hc3_test_insert_25','perturbed_text/hc3_test_insert_25.json'),
('table4','Repeat 25%',25,'hc3_test_repeat_25','perturbed_text/hc3_test_repeat_25.json'),
('table4','Replace 25%',25,'hc3_test_replace_25','perturbed_text/hc3_test_replace_25.json'),
('table4','Delete 25%',25,'hc3_test_delete_25','perturbed_text/hc3_test_delete_25.json')]
def sha(x):
 h=hashlib.sha256()
 with Path(x).open('rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
# Copy Table3/Table4 inputs to the flat original_text layout required by the author builder.
for suite,cond,noise,split,rel in CASES:
 src=a.author/rel; dst=a.author/'original_text'/(split+'.json');assert src.is_file() and sum(1 for _ in src.open(encoding='utf8'))==1000,src;
 if src.resolve()!=dst.resolve(): shutil.copyfile(src,dst)
snapshot_download('FacebookAI/roberta-base',revision='e2da8e2f811d1448a5b465c236feacd80ffbac7b',local_dir=str(a.author/'roberta-base'),local_dir_use_symlinks=False,allow_patterns=['config.json','pytorch_model.bin','vocab.json','merges.txt','tokenizer_config.json'])
s=(a.author/'building_graph.py').read_text(encoding='utf8')
s=s.replace('/root/autodl-tmp/graph_data',str(a.graphs));assert s.count('import spacy\n')==1;s=s.replace('import spacy\n','import time\nimport spacy\n',1);assert s.count('vocab_size = len(tokenizer)')==1;s=s.replace('vocab_size = len(tokenizer)',"vocab_size = len(tokenizer)\ndevice = torch.device('cuda')\nassert torch.cuda.is_available(), 'Enable GPU'\nmodel = model.to(device)\nassert not model.training",1)
assert s.count('token_embeddings = last_hidden_states[0]')==1;s=s.replace('token_embeddings = last_hidden_states[0]', 'token_embeddings = last_hidden_states[0].cpu()', 1)
old='files = [\n        "hc3_train",\n        "hc3_val",\n        "hc3_test"\n    ]';new='files = [\n'+''.join('        '+repr(x[3])+',\n' for x in CASES)+'    ]';assert s.count(old)==1;s=s.replace(old,new,1)
b=a.run/'author_building_graph_portable.py';b.write_text(s,encoding='utf8');subprocess.run([sys.executable,str(b)],cwd=a.author,check=True)
sys.path.insert(0,str(a.author));from model.GCN2 import GCN2
assert torch.cuda.is_available(),'Enable GPU';device=torch.device('cuda')
ck=[]
for x in sorted(a.ckpt.rglob('*.pth')):
 m=re.search(r'(20\d{2})(?:\.pth)?$',x.name);assert m,x;ck.append((int(m.group(1)),x))
assert len({x[0] for x in ck})==len(ck)
rows=[];detail=[]
for seed,path in ck:
 model=GCN2(768,256,64).to(device);model.load_state_dict(torch.load(path,map_location='cpu'),strict=True);model.eval();print('seed',seed,path.name,flush=True)
 for suite,cond,noise,split,rel in CASES:
  with (a.graphs/(split+'.pkl')).open('rb') as f:g=pickle.load(f)
  y=g['y'].cpu().numpy().astype(int);assert len(y)==1000 and set(y)=={0,1};prob=[]
  with torch.no_grad():
   for x,e,label in zip(g['all_token_embeddings'],g['all_edge_index'],g['y']):prob.append(model(Data(x=x,edge_index=e,y=label).to(device)).item())
  q=np.asarray(prob,dtype=np.float32);pred=(q>=.5).astype(int);cm=[[int(((y==i)&(pred==j)).sum()) for j in (0,1)] for i in (0,1)]
  r=dict(suite=suite,condition=cond,noise_percent=noise,split=split,seed=seed,checkpoint=path.name,checkpoint_sha256=sha(path),samples=1000,threshold=.5,loss=float(torch.nn.functional.binary_cross_entropy(torch.tensor(q),torch.tensor(y,dtype=torch.float32))),accuracy=float(accuracy_score(y,pred)),auc=float(roc_auc_score(y,q)),f1_human=float(f1_score(y,pred)),recall_ai=cm[0][0]/sum(cm[0]),recall_human=cm[1][1]/sum(cm[1]),confusion_matrix_rows_true_columns_pred=cm)
  rows.append(r);detail.append(dict(r,y_true=y.tolist(),p_human=q.tolist()));print(' ',split,r['accuracy'],r['auc'],flush=True)
 del model;torch.cuda.empty_cache()
metrics=['accuracy','auc','f1_human','recall_ai','recall_human'];summary=[]
for suite in ['table3','table4']:
 for cond in sorted({r['condition'] for r in rows if r['suite']==suite}):
  z=[r for r in rows if r['suite']==suite and r['condition']==cond];d=dict(suite=suite,condition=cond,num_seeds=len(z))
  for m in metrics:v=[r[m] for r in z];d[m+'_mean']=float(np.mean(v));d[m+'_std']=float(np.std(v,ddof=1)) if len(v)>1 else 0.
  summary.append(d)
noise=[r for r in rows if r['suite']=='table4' and r['noise_percent']==25];seeds=sorted({r['seed'] for r in rows});d=dict(suite='table4',condition='Average (4 noise sets)',num_seeds=len(seeds))
for m in metrics:
 v=[np.mean([r[m] for r in noise if r['seed']==s]) for s in seeds];d[m+'_mean']=float(np.mean(v));d[m+'_std']=float(np.std(v,ddof=1)) if len(v)>1 else 0.
summary.append(d)
out=a.run/'results';out.mkdir(exist_ok=True);(out/'per_seed_reports.json').write_text(json.dumps(detail,indent=2),encoding='utf8');(out/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf8')
fields=['suite','condition','noise_percent','split','seed','samples','threshold','loss']+metrics
with (out/'per_seed_metrics.csv').open('w',newline='',encoding='utf8') as f:w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows({k:r[k] for k in fields} for r in rows)
fields=['suite','condition','num_seeds']+[x for m in metrics for x in (m+'_mean',m+'_std')]
with (out/'summary.csv').open('w',newline='',encoding='utf8') as f:w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(summary)
(a.run/'run_config.json').write_text(json.dumps(dict(author_commit='83ebc6e2cd543477f83e405d702ac837e664fc9e',checkpoint_count=len(ck),threshold=.5,excluded='GPT-3.5 Mixed 0%',builder='author building_graph.py portable with RoBERTa GPU; embeddings cached on CPU',inference='author GCN2/test.py algorithm, batch 1'),indent=2),encoding='utf8')
print('finished',len(ck),'checkpoints x',len(CASES),'conditions')

In [ ]:
!{PYTHON} "{WORK}/run_eval.py" --author "{AUTHOR}" --graphs "{GRAPH}" --ckpt "{CKPT}" --run "{RUN}"
assert _exit_code == 0, 'Evaluation failed.'

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
archive=RUN/'author_table3_table4_results.zip'
with ZipFile(archive,'w',ZIP_DEFLATED) as z:
 for r in ['run_config.json','results/per_seed_reports.json','results/per_seed_metrics.csv','results/summary.json','results/summary.csv','author_building_graph_portable.py']:
  p=RUN/r;assert p.is_file(),p;z.write(p,r)
print((RUN/'results/summary.csv').read_text(encoding='utf8'))
display(FileLink(str(archive)))